# CME538 - Introduction to Data Science

## Assignment 4 - Exploratory Data Analysis

### Learning Objectives

After completing this assignment, you should be able to:

- Combine and prepare multiple CSV files for analysis.
- Work with datetime columns and timezone-aware timestamps in Pandas.
- Resample and aggregate time-series data at hourly and daily resolutions.
- Investigate missing values and make justified decisions about how to handle them.
- Identify implausible values and statistical outliers.
- Use `groupby()`, `agg()`, and resampling methods to summarize data.
- Merge datasets using aligned temporal information.
- Create clear exploratory visualizations using Matplotlib and Seaborn.
- Compare behavioural patterns across user groups.
- Investigate relationships between weather conditions and bike-share ridership.
- Interpret exploratory results while recognizing potential limitations and confounding factors.
- Write clear, concise, and reproducible analysis code.

### Submission Guidelines

You may add additional cells for exploratory or scratch work while completing the assignment. Before submitting:

- Remove unnecessary scratch cells and outputs.
- Keep only the code and output needed to answer each question.
- Do not display entire DataFrames or unnecessarily large outputs.
- Use the variable names specified in each question.
- Do not modify cells labelled `# Verification - do not modify`.
- Complete all designated written-answer cells.
- Restart the kernel and run the notebook from beginning to end to ensure that all cells execute without errors.

# Marking Breakdown

| Component | Marks |
|---|---:|
| Questions 1–6: Data preparation and time series | 6 |
| Questions 7–12: Missing data and data cleaning | 6 |
| Questions 13–16: Data merging and daily ridership analysis | 5 |
| Questions 17–20: Rider behaviour and interpretation | 5 |
| Questions 21–25: Weather and ridership analysis | 5 |
| Code quality | 3 |
| **Total** | **30** |

### Code Quality

Code quality will be assessed across the complete notebook.

| Level | Points | Description |
|---|---:|---|
| **Developing** | 1 | Code produces the required results but may be difficult to follow, unnecessarily repetitive, poorly organized, or include excessive output. |
| **Competent** | 2 | Code is organized and readable, uses appropriate Python and Pandas operations, and produces concise, relevant outputs. |
| **Strong** | 3 | Code is clear, concise, well organized, and reproducible; uses Python and Pandas effectively; avoids unnecessary operations and output; and executes successfully from beginning to end. |

## Notebook Setup

In [ ]:
# Import required libraries
import os

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Overview

## Your Assignment: Bike Share Toronto

You have joined a data analytics team supporting the **City of Toronto**.

Bike Share Toronto has provided trip records for 2019, and the team wants to better understand how, when, and under what conditions people use the system. The city is particularly interested in differences between **Annual Members** and **Casual Members**, as well as the relationship between weather and ridership.

There is one complication: the information you need is spread across multiple datasets.

Bike-share trips are stored in monthly CSV files, while weather observations are recorded separately by the **Toronto City Centre** weather station. The datasets also contain some of the issues commonly encountered in real-world data: missing values, duplicate records, unusual trip durations, different temporal resolutions, and datetime information that must be handled carefully.

### Your Task

Your team has asked you to prepare the data and investigate three questions:

1. **When do people ride?**  
   Explore how bike-share activity changes across days and hours.

2. **Who rides differently?**  
   Compare riding patterns between **Annual Members** and **Casual Members**.

3. **Does weather matter?**  
   Investigate how temperature and weather conditions are associated with ridership.

To answer these questions, you will build an exploratory data analysis workflow that takes you from raw files to interpretable results.

By the end of the assignment, you will have transformed multiple raw datasets into a cleaned, merged, and analysis-ready dataset—and used it to tell a data-supported story about bike-share activity in Toronto.

---

# 1. Preparing the Weather Data

Before the team can investigate ridership, you need to prepare the weather observations that will eventually be matched to bike-share trips.

## Question 1 - Find the Weather Files

The assignment directory contains monthly hourly-weather files for 2019 from the **Toronto City Centre** weather station.

The filenames contain the station identifier `6158359` and end with `.csv`.

Create a variable named `weather_filenames` containing only these weather files.

Sort the filenames alphabetically so that the result is reproducible.

In [ ]:
# Find all files in the current directory that:
# 1. contain the weather station ID "6158359"
# 2. end with ".csv"
# Sort the resulting filenames alphabetically.

weather_filenames = ...

# Display the first five filenames
weather_filenames[:5]

In [ ]:
# Verification - do not modify
print(f"Q1 Answer - Number of weather files: {len(weather_filenames)}")
print(f"First file: {weather_filenames[0]}")
print(f"Last file: {weather_filenames[-1]}")

---

## Question 2 - Combine the Weather Data

The files in `weather_filenames` contain monthly hourly-weather observations for 2019.

Read each CSV file into Pandas and combine all monthly datasets into a single DataFrame named `weather_data`.

While importing the files, parse the `Date/Time` column as datetime values.

Use `pd.concat()` to combine the monthly DataFrames and reset the row index.

In [ ]:
# Read each weather CSV file.
# Hint: use parse_dates=["Date/Time"] when calling pd.read_csv().

weather_frames = ...

# Combine the DataFrames into one DataFrame.
# Use ignore_index=True to create a new continuous index.
weather_data = ...

weather_data.head()

In [ ]:
# Verification - do not modify
print(f"Q2 Answer - Number of rows: {weather_data.shape[0]}")
print(f"Number of columns: {weather_data.shape[1]}")
print(f"Date/Time dtype: {weather_data['Date/Time'].dtype}")

---

## Question 3 - Prepare and Visualize the Weather Time Series

Prepare `weather_data` for time-series analysis.

1. Set `Date/Time` as the DataFrame index.
2. Localize the datetime index to the `America/Toronto` time zone.
3. Sort the DataFrame chronologically.

Because daylight saving time can create ambiguous or nonexistent local timestamps, use:

- `ambiguous="NaT"`
- `nonexistent="NaT"`

After localization, remove any rows whose datetime index is missing.

Finally, create a line plot showing `Temp (°C)` across 2019.

In [ ]:
# Set "Date/Time" as the DataFrame index.
weather_data = ...

# Localize the index to "America/Toronto".
# Use ambiguous="NaT" and nonexistent="NaT".
weather_data.index = ...

# Remove rows with missing timestamps created during localization.
weather_data = ...

# Sort the DataFrame by its datetime index.
weather_data = ...

weather_data.head()

In [ ]:
# Plot temperature across 2019.
# Use the datetime index for x and "Temp (°C)" for y.

plt.figure(figsize=(10, 5))

# YOUR CODE HERE

plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.title("Hourly Temperature in Toronto - 2019")
plt.tight_layout()
plt.show()

In [ ]:
# Verification - do not modify
print(f"Q3 Answer - Index type: {type(weather_data.index).__name__}")
print(f"Time zone: {weather_data.index.tz}")
print(f"Missing timestamps: {weather_data.index.isna().sum()}")
print(f"Index sorted: {weather_data.index.is_monotonic_increasing}")

---

# 2. Preparing the Bike Share Data

Toronto's Bike Share system records individual trips throughout the year. The assignment directory contains one CSV file for each month of 2019, with filenames following the pattern:

`bike_share_YYYY-MM.csv`

Before investigating ridership patterns, we first need to combine the monthly files and prepare the trip timestamps for time-series analysis.

---

## Question 4 - Combining the Bike Share Files

Identify the monthly Bike Share files in the assignment directory.

Create a variable named `trips_filenames` containing all filenames that begin with `bike_share_` and end with `.csv`. Sort the filenames alphabetically so that the months are processed in chronological order.

Then, read all monthly files and combine them into a single DataFrame named `trips_data`.

Some of the original column names contain inconsistent whitespace. Clean the column names so that consecutive spaces are replaced by a single space.

In [ ]:
# Find files that begin with "bike_share_" and end with ".csv".
# Sort the filenames alphabetically.
trips_filenames = ...

# Read each CSV and combine them into one DataFrame.
trips_data = ...

# Clean the column names by replacing consecutive whitespace
# with a single space.
trips_data.columns = ...

trips_data.head()

In [ ]:
# Verification - do not modify
print(f"Q4 Answer - Number of trip files: {len(trips_filenames)}")
print(f"Number of rows: {trips_data.shape[0]}")
print(f"Number of columns: {trips_data.shape[1]}")
print(f"First file: {trips_filenames[0]}")
print(f"Last file: {trips_filenames[-1]}")

---

## Question 5 - Preparing Trip Datetimes

Each trip contains a `Start Time` and an `End Time`. These columns are initially imported as text and must be converted to datetime values before they can be used for time-series analysis.

Convert both columns to Pandas datetime values and localize them to the `America/Toronto` time zone.

Using `America/Toronto` rather than a fixed `EST` offset accounts for both Eastern Standard Time and Eastern Daylight Time during the year.

Daylight saving time transitions can create ambiguous or nonexistent local timestamps. Handle these cases using:

- `ambiguous="NaT"`
- `nonexistent="NaT"`

After localization, remove records with missing `Start Time` or `End Time` values.

In [ ]:
# Convert "Start Time" and "End Time" to datetime values.
trips_data["Start Time"] = ...
trips_data["End Time"] = ...

# Localize both columns to "America/Toronto".
# Use ambiguous="NaT" and nonexistent="NaT".
trips_data["Start Time"] = ...
trips_data["End Time"] = ...

# Remove rows where either timestamp is missing.
trips_data = ...

trips_data.head()

In [ ]:
# Verification - do not modify
print(f"Q5 Answer - Start Time dtype: {trips_data['Start Time'].dtype}")
print(f"End Time dtype: {trips_data['End Time'].dtype}")
print(f"Missing Start Time values: {trips_data['Start Time'].isna().sum()}")
print(f"Missing End Time values: {trips_data['End Time'].isna().sum()}")
print(f"Rows remaining: {len(trips_data)}")

---

## Question 6 - Visualizing Daily Ridership

Before continuing with the analysis, perform a visual sanity check of the trip timestamps.

Using `Start Time`, calculate the total number of Bike Share trips recorded on each day of 2019. Use Pandas time-series functionality such as `.resample()` to create a daily ride-count series named `daily_rides`.

Create a line plot showing how daily ridership changed throughout the year.

Your plot should:

- display date on the x-axis,
- display the number of rides on the y-axis,
- include an appropriate title, and
- show the complete 2019 ridership pattern.

In [ ]:
# Set "Start Time" as the index and resample by day.
# Count the number of trips in each day.
daily_rides = ...

# Plot daily ride counts across 2019.
plt.figure(figsize=(10, 5))

# YOUR CODE HERE

plt.xlabel("Date")
plt.ylabel("Number of Rides")
plt.title("Daily Bike Share Rides - 2019")
plt.tight_layout()
plt.show()

In [ ]:
# Verification - do not modify
print(f"Q6 Answer - Number of daily observations: {len(daily_rides)}")
print(f"Total rides represented: {daily_rides.sum()}")
print(f"Maximum daily rides: {daily_rides.max()}")
print(f"Date of maximum rides: {daily_rides.idxmax()}")

---

# 3. Investigating Missing Data

Real-world datasets are rarely complete. Missing values may result from data collection problems, unavailable measurements, or the way information is encoded.

Before removing any observations, it is important to understand **where values are missing and what those missing values mean**.

In this section, we will examine missingness in both the weather and Bike Share datasets.

----

## Question 7 - Investigating Missing Values

Calculate the number of missing values in each column of `weather_data`.

Create a DataFrame named `weather_data_missing` with:

- the original column names as the index, and
- one column named `count` containing the number of missing values.

Repeat the same analysis for `trips_data` and store the result in a DataFrame named `trips_data_missing`.

Sort both results from the largest to the smallest number of missing values so that the most affected variables are easy to identify.

In [ ]:
# Count missing values in each weather column.
# Sort from largest to smallest and convert the result
# to a DataFrame with a column named "count".
weather_data_missing = ...

weather_data_missing

In [ ]:
# Repeat the missing-value analysis for the trip data.
trips_data_missing = ...

trips_data_missing

In [ ]:
# Verification - do not modify
print("Q7 Answer - Weather columns with missing values:")
print(weather_data_missing.query("count > 0"))

print("\nTrip columns with missing values:")
print(trips_data_missing.query("count > 0"))

### Interpreting Missing Weather Data

Missing values do not always represent errors.

Consider the `Weather` column. Environment and Climate Change Canada records descriptions such as rain, snow, fog, and other notable weather conditions. Hours without a reported weather event may therefore contain a missing value in this field.

Examine the non-missing values recorded in `Weather`.

In [ ]:
# Examine reported weather conditions
weather_data["Weather"].dropna().value_counts()

For this analysis, missing values in `Weather` should **not** be removed automatically. They will later be used to help distinguish hours without a reported precipitation/weather event from hours with reported conditions.

Other weather variables may also contain missing observations, so missingness should be handled according to the variables required for a particular analysis rather than by dropping every incomplete weather row.

## Question 8 - Cleaning Missing Trip Data

For this assignment, treat any trip record containing a missing value as incomplete and remove it.

Determine how many trip records contain at least one missing value.

Then remove rows containing missing values from `trips_data`.

Report:

- the number of rows before cleaning,
- the number of rows removed, and
- the number of rows remaining.

In [ ]:
# Record the number of rows before cleaning.
rows_before = ...

# Remove every trip containing at least one missing value.
trips_data = ...

# Calculate how many rows were removed.
rows_removed = ...

trips_data.head()

In [ ]:
# Verification - do not modify
print(f"Q8 Answer - Rows before cleaning: {rows_before}")
print(f"Rows removed: {rows_removed}")
print(f"Rows remaining: {len(trips_data)}")
print(f"Missing values remaining: {trips_data.isna().sum().sum()}")

----

# 4. Cleaning Trip Duration and Duplicate Records

Large datasets often contain unusual observations. Some may represent genuine extreme behaviour, while others may result from data-entry errors, system artifacts, or incomplete trips.

Before removing outliers, we should first inspect the distribution of the data and apply cleaning rules that have a clear justification.

----

## Question 9 - Identifying Implausible Trip Durations

Use `.describe()` to inspect the summary statistics of the numeric columns in `trips_data`.

Pay particular attention to `Trip Duration`, which is measured in seconds.

Bike Share Toronto considers trips lasting less than **60 seconds** to be false or invalid trips.

Remove all records with a `Trip Duration` less than 60 seconds.

In [ ]:
# Examine summary statistics
trips_data.describe()

In [ ]:
# Record the number of rows before filtering.
rows_before_duration_filter = ...

# Keep only trips lasting at least 60 seconds.
trips_data = ...

# Calculate how many short trips were removed.
short_trips_removed = ...

trips_data.head()

In [ ]:
# Verification - do not modify
print(f"Q9 Answer - Trips shorter than 60 seconds removed: {short_trips_removed}")
print(f"Minimum remaining trip duration: {trips_data['Trip Duration'].min():.0f} seconds")
print(f"Rows remaining: {len(trips_data)}")

----

## Question 10 - Removing Trip-Duration Outliers

Even after removing trips shorter than one minute, unusually long trips may remain.

Use the **interquartile range (IQR)** method to identify outliers in `Trip Duration`.

Calculate:

- `Q1`: the 25th percentile,
- `Q3`: the 75th percentile,
- `IQR = Q3 - Q1`,
- lower bound: `Q1 - 1.5 × IQR`, and
- upper bound: `Q3 + 1.5 × IQR`.

Remove trips with durations outside these bounds.

Calculate the thresholds using the dataset **after** the minimum 60-second rule from Question 9 has been applied.

In [ ]:
# Calculate the first and third quartiles of "Trip Duration".
q1 = ...
q3 = ...

# Calculate the interquartile range.
iqr = ...

# Calculate the lower and upper IQR bounds.
lower_bound = ...
upper_bound = ...

# Record the number of rows before filtering.
rows_before_iqr_filter = ...

# Keep trips whose duration falls between the two bounds.
# Hint: Series.between() may be useful.
trips_data = ...

# Calculate how many outliers were removed.
iqr_outliers_removed = ...

trips_data.head()

In [ ]:
# Verification - do not modify
print(f"Q10 Answer - Q1: {q1:.2f} seconds")
print(f"Q3: {q3:.2f} seconds")
print(f"IQR: {iqr:.2f} seconds")
print(f"Lower bound: {lower_bound:.2f} seconds")
print(f"Upper bound: {upper_bound:.2f} seconds")
print(f"IQR outliers removed: {iqr_outliers_removed}")
print(f"Rows remaining: {len(trips_data)}")

----

## Question 11 - Visualizing the Cleaned Trip-Duration Distribution

Now examine the distribution of trip durations after cleaning.

Create a new variable named `trip_duration_minutes` containing `Trip Duration` converted from seconds to minutes.

Use `sns.histplot()` to create a histogram of trip durations and display a KDE curve on the same figure.

Your plot should:

- display trip duration in **minutes** on the x-axis,
- display frequency on the y-axis,
- include the histogram and KDE curve, and
- include an appropriate title.

In [ ]:
# Convert trip duration from seconds to minutes.
trip_duration_minutes = ...

# Create a histogram with a KDE curve.
plt.figure(figsize=(8, 5))

sns.histplot(
    trip_duration_minutes,
    bins=40,
    kde=True
)

plt.xlabel("Trip Duration (minutes)")
plt.ylabel("Number of Trips")
plt.title("Distribution of Bike Share Trip Durations")
plt.tight_layout()
plt.show()

In [ ]:
# Verification - do not modify
print(f"Q11 Answer - Number of trips plotted: {trip_duration_minutes.shape[0]}")
print(f"Minimum duration: {trip_duration_minutes.min():.2f} minutes")
print(f"Median duration: {trip_duration_minutes.median():.2f} minutes")
print(f"Maximum duration: {trip_duration_minutes.max():.2f} minutes")

----

## Question 12 - Removing Duplicate Trips

A unique `Trip Id` should represent a single Bike Share trip. Duplicate identifiers may therefore indicate that the same trip record appears more than once in the dataset.

Determine how many duplicated `Trip Id` values are present.

Then remove duplicate trips based on `Trip Id`, keeping the first occurrence of each trip.

In [ ]:
# Count duplicated "Trip Id" values.
duplicate_trip_ids = ...

# Remove duplicates based on "Trip Id".
# Keep the first occurrence.
trips_data = ...

trips_data.head()

In [ ]:
# Verification - do not modify
print(f"Q12 Answer - Duplicate trips removed: {duplicate_trip_ids}")
print(f"Duplicate Trip Id values remaining: {trips_data['Trip Id'].duplicated().sum()}")
print(f"Rows remaining: {len(trips_data)}")

----

# 5. Merging Bike Share and Weather Data

So far, the Bike Share trips and weather observations have been stored separately.

To investigate whether weather conditions are associated with ridership, we need to connect each trip with the weather conditions around the time that trip began.

The Bike Share data record exact trip start times, while the weather data are reported hourly. We therefore need to create a common hourly timestamp before merging the datasets.

---

## Question 13 - Merging Trips with Weather Data

Create a new column in `trips_data` named `merge_time` based on `Start Time`, floored to the beginning of the hour.

For example, a trip beginning at `10:47` should receive a `merge_time` of `10:00`.

Then use `.merge()` to combine `trips_data` with `weather_data`.

Store the resulting DataFrame in a variable named `data_merged`.

Use the trip's start time to determine which hourly weather observation should be associated with each ride.

In [ ]:
# Create "merge_time" by flooring each Start Time to the hour.
# Example: 10:47 becomes 10:00.
trips_data["merge_time"] = ...

# Merge trips with weather observations.
# Match merge_time to the weather DataFrame's datetime index.
# Keep all trips, and validate that many trips can match one weather row.
data_merged = ...

data_merged.head()

In [ ]:
# Verification - do not modify
print(f"Q13 Answer - Rows before merge: {len(trips_data)}")
print(f"Rows after merge: {len(data_merged)}")
print(f"Missing temperature values after merge: {data_merged['Temp (°C)'].isna().sum()}")
print(f"Duplicate Trip Id values after merge: {data_merged['Trip Id'].duplicated().sum()}")

----

# 6. Exploring Rider Behaviour

Bike Share Toronto serves two broad types of riders:

- **Annual Members**, who purchase a membership and may use Bike Share as part of their regular transportation routine.
- **Casual Members**, who pay on a shorter-term or per-use basis and may use Bike Share more recreationally.

In this section, we will compare these groups across days of the week and times of day.

The goal is to determine whether their riding patterns suggest different types of travel behaviour.

## Question 14 - Daily Ridership by User Type

Create a DataFrame named `data_days` containing one row for each day.

It should contain four columns:

- `rides`: total number of rides that day,
- `annual_members`: number of rides by Annual Members,
- `casual_members`: number of rides by Casual Members, and
- `workday`: `True` for Monday through Friday and `False` for Saturday and Sunday.

Use `Start Time` to determine the date of each trip.

Hint: Consider creating temporary indicator columns for Annual and Casual Member rides, then use `.groupby()` and `.agg()` to calculate the daily totals.

In [ ]:
# Create 0/1 indicator columns for Annual and Casual Member rides.
data_merged["annual_member"] = ...
data_merged["casual_member"] = ...

# Group trips by calendar day and calculate:
# - total rides
# - Annual Member rides
# - Casual Member rides
data_days = (
    data_merged
    .groupby(...)
    .agg(
        rides=(..., ...),
        annual_members=(..., ...),
        casual_members=(..., ...)
    )
)

# Monday=0, ..., Sunday=6.
# Mark Monday-Friday as workdays.
data_days["workday"] = ...

data_days.head()

In [ ]:
# Verification - do not modify
print(f"Q14 Answer - Number of days: {len(data_days)}")
print(f"Total rides represented: {data_days['rides'].sum()}")
print(f"Rows in merged data: {len(data_merged)}")
print(
    "Rider totals match:",
    (
        data_days["annual_members"]
        + data_days["casual_members"]
        == data_days["rides"]
    ).all()
)
print(f"Workdays: {data_days['workday'].sum()}")
print(f"Weekend days: {(~data_days['workday']).sum()}")

----

## Question 15 - Comparing Daily Ridership Distributions

Compare the distributions of daily ride counts for Annual Members and Casual Members.

Create a single figure containing both distributions.

Use `sns.histplot()` with a KDE curve rather than the deprecated `sns.distplot()`.

Your plot should:

- show both rider types,
- use daily ride count on the x-axis,
- include a histogram and density curve,
- clearly distinguish the two rider types,
- include appropriate axis labels, a title, and a legend.

In [ ]:
# Plot both daily ridership distributions on the same figure.
# Use sns.histplot() with kde=True for each rider type.
# Hint: use stat="density" so both histograms are comparable.

plt.figure(figsize=(8, 5))

# Annual Members
# YOUR CODE HERE

# Casual Members
# YOUR CODE HERE

plt.xlabel("Daily Number of Rides")
plt.ylabel("Density")
plt.title("Distribution of Daily Ridership by User Type")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Verification - do not modify
print("Q15 Answer - Daily ridership summary:")
print(
    data_days[
        ["annual_members", "casual_members"]
    ].describe().loc[["mean", "50%", "max"]].round(1)
)

----

## Question 16 - Relationship Between Annual and Casual Ridership

Now investigate whether days with high Annual Member ridership also tend to have high Casual Member ridership.

Create a scatter plot with:

- `annual_members` on the x-axis,
- `casual_members` on the y-axis, and
- points distinguished according to whether the day is a workday or weekend.

Include appropriate labels, a title, and a legend.

In [ ]:
# Create a scatter plot comparing Annual and Casual Member rides.
# Use "workday" to distinguish the points.

plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=data_days,
    x=...,
    y=...,
    hue=...
)

plt.xlabel("Annual Member Rides")
plt.ylabel("Casual Member Rides")
plt.title("Daily Annual vs. Casual Member Ridership")
plt.tight_layout()
plt.show()

In [ ]:
# Verification - do not modify
print("Q16 Answer - Average daily rides by workday status:")
print(
    data_days
    .groupby("workday")[["annual_members", "casual_members"]]
    .mean()
    .round(1)
)

----

## Question 17 - Interpreting Unusual Workday Patterns

Look closely at your scatter plot from Question 16.

Some dates classified as `workday = True` may behave more like weekends, with relatively fewer Annual Member rides and relatively more Casual Member rides.

What could explain these observations?

Identify at least one plausible explanation and describe one additional piece of information that could be added to the dataset to investigate your hypothesis.

*Type your answer below.*

----

## Question 18 - Average Ridership by Hour of Day

Daily totals can hide important patterns within the day.

Create a DataFrame named `data_hours` with an index representing the hour of day from `0` to `23`.

The DataFrame should contain:

- `rides`: average total number of rides,
- `annual_members`: average number of Annual Member rides, and
- `casual_members`: average number of Casual Member rides.

The averages should represent the typical number of rides occurring during each hour of the day across 2019.

Hint: Resample the ride-level data to an hourly frequency first. This ensures that zero-ride hours between the first and last recorded trips are retained before calculating the average for each hour of day.

In [ ]:
# Step 1: Resample the ride-level data to hourly counts.
# This is important because it retains hours with zero rides.
rides_by_hour = (
    data_merged
    .set_index("Start Time")
    .resample("h")
    .agg(
        rides=(..., ...),
        annual_members=(..., ...),
        casual_members=(..., ...)
    )
)

# Step 2: Group the hourly observations by hour of day (0-23)
# and calculate the average for each hour.
data_hours = ...

data_hours.index.name = "hour"

data_hours.head()

In [ ]:
# Verification - do not modify
print(f"Q18 Answer - Number of hourly observations: {len(data_hours)}")
print(f"Hour range: {data_hours.index.min()} to {data_hours.index.max()}")

print("\nPeak hour by rider type:")
print(
    data_hours[
        ["annual_members", "casual_members"]
    ].idxmax()
)

----

## Question 19 - Visualizing Hourly Ridership Patterns

Use `data_hours` to compare how Annual Members and Casual Members use Bike Share throughout the day.

Create a line plot showing:

- hour of day on the x-axis,
- average number of rides on the y-axis,
- one line for Annual Members, and
- one line for Casual Members.

Display the complete 24-hour period and include appropriate labels, a title, and a legend.

In [ ]:
# Plot average hourly ridership for both rider types.

plt.figure(figsize=(9, 5))

# Plot Annual Members
# YOUR CODE HERE

# Plot Casual Members
# YOUR CODE HERE

plt.xlim(0, 23)
plt.xticks(range(0, 24, 2))
plt.xlabel("Hour of Day")
plt.ylabel("Average Number of Rides")
plt.title("Average Hourly Ridership by User Type")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Verification - do not modify
print("Q19 Answer - Peak riding hour:")
print(f"Annual Members: {data_hours['annual_members'].idxmax()}:00")
print(f"Casual Members: {data_hours['casual_members'].idxmax()}:00")

print("\nMaximum average hourly rides:")
print(
    data_hours[
        ["annual_members", "casual_members"]
    ].max().round(1)
)

----

## Question 20 - Interpreting Hourly Riding Patterns

Compare the hourly patterns for Annual Members and Casual Members in Question 19.

Describe the main differences you observe between the two groups.

In particular:

1. Identify any important peaks in ridership.
2. Explain what those peaks might suggest about how each type of rider uses Bike Share Toronto.

*Type your answer below.*

----

# 7. Exploring Weather and Ridership

The final part of the analysis investigates how weather conditions are associated with Bike Share activity.

The merged dataset contains ride-level observations together with hourly weather information. Because many rides can occur during the same hour, the weather values are repeated across multiple rows.

Before analyzing weather and ridership, we will first aggregate the data to an hourly resolution and then summarize it by day.

One important detail concerns the `Weather` column. A missing value does not necessarily indicate missing sensor data. Instead, it generally means that no notable weather phenomenon was reported for that hour.

Let's inspect the weather descriptions that are present.

In [ ]:
# Inspect reported weather conditions
weather_conditions = (
    weather_data["Weather"]
    .dropna()
    .value_counts()
)

weather_conditions.head(15)

For this analysis, we will distinguish between hours with a reported **precipitation event** and hours without one.

We will treat weather descriptions containing terms such as `Rain`, `Snow`, `Drizzle`, or `Thunderstorm` as precipitation events.

Other reported conditions, such as fog, are not automatically classified as precipitation.

----

## Question 21 - Creating an Hourly Analysis Dataset

The current `data_merged` DataFrame contains one row for each Bike Share trip. To compare hourly ridership with hourly weather conditions, aggregate the data so that each row represents one hour.

Create a DataFrame named `hourly_rides_and_weather` with a timezone-aware hourly DatetimeIndex and the following columns:

- `rides` - total number of rides during the hour,
- `annual_members` - number of Annual Member rides,
- `casual_members` - number of Casual Member rides,
- `workday` - `True` for Monday through Friday and `False` for Saturday and Sunday,
- `temp` - hourly temperature in degrees Celsius, and
- `weather` - the reported weather description for that hour.

First aggregate the Bike Share trips to an hourly frequency using `Start Time`.

Then combine the hourly ride counts with the complete hourly weather timeline from `weather_data`. Hours with no recorded Bike Share trips should remain in the dataset with ride counts of `0`.

In [ ]:
# Step 1: Aggregate individual trips to hourly ride counts.
hourly_rides = (
    data_merged
    .set_index("Start Time")
    .resample("h")
    .agg(
        rides=(..., ...),
        annual_members=(..., ...),
        casual_members=(..., ...)
    )
)

# Step 2: Select temperature and weather description.
# Rename them to "temp" and "weather".
hourly_weather = ...

# Step 3: Join ride counts onto the complete weather timeline.
# The weather timeline should determine which hours are retained.
hourly_rides_and_weather = ...

# Step 4: Hours without trips should have ride counts of 0.
ride_columns = ["rides", "annual_members", "casual_members"]

hourly_rides_and_weather[ride_columns] = ...

# Step 5: Add a Monday-Friday workday indicator.
hourly_rides_and_weather["workday"] = ...

# Keep the columns in the required order.
hourly_rides_and_weather = hourly_rides_and_weather[
    [
        "rides",
        "annual_members",
        "casual_members",
        "workday",
        "temp",
        "weather"
    ]
]

hourly_rides_and_weather.head(10)

In [ ]:
# Verification - do not modify
print(f"Q21 Answer - Number of hourly observations: {len(hourly_rides_and_weather)}")
print(f"Index type: {type(hourly_rides_and_weather.index).__name__}")
print(f"Time zone: {hourly_rides_and_weather.index.tz}")
print(f"Hours with zero rides: {(hourly_rides_and_weather['rides'] == 0).sum()}")

print(
    "Rider totals match:",
    (
        hourly_rides_and_weather["annual_members"]
        + hourly_rides_and_weather["casual_members"]
        == hourly_rides_and_weather["rides"]
    ).all()
)

----

## Question 22 - Creating a Daily Weather and Ridership Dataset

Next, aggregate the hourly data to a daily resolution.

Create a new DataFrame named `daily_rides_and_weather` with one row for each day and the following columns:

- `rides` - total number of rides that day,
- `annual_members` - total Annual Member rides,
- `casual_members` - total Casual Member rides,
- `workday` - whether the day is Monday through Friday,
- `temp` - maximum temperature recorded that day, and
- `weather` - either `No Precipitation` or `Precipitation`.

To create the daily `weather` category:

1. identify whether each hourly weather description contains a precipitation-related term, and
2. classify the day as `Precipitation` if **at least one hourly observation** contains a precipitation event.

Otherwise, classify the day as `No Precipitation`.

For this assignment, treat the following terms as indicators of precipitation:

- `Rain`
- `Snow`
- `Drizzle`
- `Thunderstorm`
- `Freezing Rain`
- `Ice Pellets`

In [ ]:
# These terms indicate precipitation.
precipitation_pattern = (
    "Rain|Snow|Drizzle|Thunderstorm|Freezing Rain|Ice Pellets"
)

# Create a Boolean column indicating whether precipitation
# was reported during each hour.
# Hint: fill missing weather descriptions with "" before str.contains().
hourly_rides_and_weather["precipitation"] = ...

# Aggregate the hourly data to daily values.
# Sum ride counts, take the maximum temperature, and use
# the maximum Boolean value to determine whether precipitation
# occurred at least once during the day.
daily_rides_and_weather = (
    hourly_rides_and_weather
    .resample("D")
    .agg(
        rides=(..., ...),
        annual_members=(..., ...),
        casual_members=(..., ...),
        temp=(..., ...),
        precipitation=(..., ...)
    )
)

# Add the Monday-Friday workday indicator.
daily_rides_and_weather["workday"] = ...

# Convert True/False precipitation values to the required labels.
daily_rides_and_weather["weather"] = ...

# Remove the temporary precipitation column.
daily_rides_and_weather = ...

# Put the columns in the required order.
daily_rides_and_weather = daily_rides_and_weather[
    [
        "rides",
        "annual_members",
        "casual_members",
        "workday",
        "temp",
        "weather"
    ]
]

daily_rides_and_weather.head(10)

In [ ]:
# Verification - do not modify
print(f"Q22 Answer - Number of daily observations: {len(daily_rides_and_weather)}")
print(
    f"Total rides represented: "
    f"{daily_rides_and_weather['rides'].sum()}"
)
print("Weather categories:")
print(daily_rides_and_weather["weather"].value_counts())
print(
    f"Temperature range: "
    f"{daily_rides_and_weather['temp'].min():.1f} to "
    f"{daily_rides_and_weather['temp'].max():.1f} °C"
)

----

## Question 23 - Ridership and Precipitation

Investigate whether daily ridership differs between days classified as `Precipitation` and `No Precipitation`.

Create a violin plot using `sns.violinplot()` with:

- daily weather category on the x-axis,
- total daily rides on the y-axis,
- an appropriate title and axis labels.

The figure should make it easy to compare the distribution of daily ridership under the two weather conditions.

In [ ]:
# Compare daily ridership on precipitation and non-precipitation days.

plt.figure(figsize=(8, 5))

sns.violinplot(
    data=daily_rides_and_weather,
    x=...,
    y=...,
    inner="quartile"
)

plt.xlabel("Weather Condition")
plt.ylabel("Daily Number of Rides")
plt.title("Daily Bike Share Ridership by Weather Condition")
plt.tight_layout()
plt.show()

In [ ]:
# Verification - do not modify
print("Q23 Answer - Daily ridership by weather category:")
print(
    daily_rides_and_weather
    .groupby("weather")["rides"]
    .agg(["count", "mean", "median"])
    .round(1)
)

----

## Question 24 - Ridership and Temperature

Now investigate the relationship between daily maximum temperature and total daily ridership.

Create a scatter plot using `sns.scatterplot()` with:

- maximum daily temperature on the x-axis,
- total daily rides on the y-axis,
- points distinguished by `workday`,
- an appropriate title and axis labels.

Use the figure to examine whether warmer days tend to be associated with greater Bike Share activity and whether the relationship differs between workdays and weekends.

In [ ]:
# Plot daily rides against maximum daily temperature.
# Distinguish workdays from weekends.

plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=daily_rides_and_weather,
    x=...,
    y=...,
    hue=...
)

plt.xlabel("Maximum Daily Temperature (°C)")
plt.ylabel("Daily Number of Rides")
plt.title("Daily Bike Share Ridership vs. Temperature")
plt.tight_layout()
plt.show()

In [ ]:
# Verification - do not modify
temperature_correlation = (
    daily_rides_and_weather[["temp", "rides"]]
    .corr()
    .loc["temp", "rides"]
)

print(
    f"Q24 Answer - Correlation between maximum temperature "
    f"and daily rides: {temperature_correlation:.3f}"
)

print("\nAverage rides by workday status:")
print(
    daily_rides_and_weather
    .groupby("workday")["rides"]
    .mean()
    .round(1)
)

---

## Question 25 - Interpreting Weather and Ridership

Reflect on the figures from Questions 23 and 24.

Discuss the main patterns you observe between:

1. precipitation and daily ridership, and
2. temperature and daily ridership.

Then identify at least one limitation of these comparisons.

Consider whether variables such as **season, day of week, holidays, daylight hours, or interactions between weather conditions** could influence the relationships you observe.

Finally, suggest one improvement or additional variable that could make the analysis more informative.

*Type your answer below.*

----

# Submission

Once you have completed the assignment:

1. Restart the kernel and run all cells to ensure the notebook executes without errors.
2. Save your completed notebook (`.ipynb`).
3. Export the notebook as an HTML file (`.html`).
4. Submit **both the `.ipynb` and `.html` files** to Quercus.

*Note: Sample figures showing the expected general appearance of the plots are available in the images folder. Your figures do not need to match these examples exactly, but they should display the overall patterns clearly.*